# Greenwashing Detection Pipeline
**MSIN0166 Data Engineering Group Assignment**

## Pipeline Overview
This notebook builds an end-to-end data engineering pipeline to detect potential greenwashing 
among FTSE 100 companies by comparing self-reported ESG claims against independent data sources.

### Data Sources
| # | Source | Type | Storage |
|---|--------|------|---------|
| 1 | Company sustainability websites | Web scraping | MongoDB + Parquet |
| 2 | The Guardian API | News API | MongoDB + Parquet |
| 3 | Reddit Public JSON | Social media | MongoDB + Parquet |
| 4 | Our World in Data CO2 | Open dataset | Parquet |
| 5 | NetworkX Graph | Relationship model | GEXF |

### Processing Stack
- **Apache Spark** — distributed joins, aggregations, derived columns
- **DuckDB** — analytical SQL queries on Parquet files
- **MongoDB** — NoSQL storage for unstructured text

---
## Section 0: Setup & Imports
Load all libraries and API keys from the `.env` file. 
Create output directories for raw and processed data.

In [2]:
# Run this cell ONCE to install all pipeline dependencies into the active kernel
import subprocess, sys

packages = [
    "requests", "beautifulsoup4", "pandas", "pyarrow", "fastparquet",
    "pymongo", "pyspark", "duckdb", "newsapi-python", "python-dotenv",
    "networkx", "streamlit", "plotly", "nbdime"
]

print("Installing packages...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet"] + packages,
    capture_output=True, text=True
)
if result.returncode == 0:
    print("All packages installed successfully. Now run the next cell.")
else:
    print("Some packages failed to install:")
    print(result.stderr[-1000:])

Installing packages...
All packages installed successfully. Now run the next cell.


In [4]:
import os, re, json, time, requests
import pandas as pd
import pymongo
import pyarrow.parquet as pq
import duckdb
import networkx as nx
from datetime import datetime
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import (col, when, lit, count, avg, 
                                    round as spark_round, sum as spark_sum,
                                    dense_rank, desc)
from pyspark.sql.window import Window

load_dotenv()
NEWS_API_KEY            = os.getenv("NEWS_API_KEY")
GUARDIAN_API_KEY        = os.getenv("https://content.guardianapis.com/search")
MONGO_URI               = os.getenv("MONGO_URI", "mongodb://localhost:27017/")
COMPANIES_HOUSE_API_KEY = os.getenv("COMPANIES_HOUSE_API_KEY", "")

os.makedirs("data/raw",       exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

# Clear lineage log at start of fresh run
with open("data/lineage_log.jsonl", "w") as f:
    pass

print("Imports successful")
print(f"MongoDB URI      : {MONGO_URI}")

Imports successful
MongoDB URI      : mongodb://localhost:27017/


### Lineage Logger
Every data collection step writes a structured entry to `data/lineage_log.jsonl`.
This creates a full, reproducible audit trail — a core data engineering best practice.
Each entry records: source name, URL, timestamp, record count, output path, and transformations applied.

In [5]:
LINEAGE_FILE = "data/lineage_log.jsonl"

def log_lineage(source_name, source_url, record_count, output_path, transformations=None):
    entry = {
        "source":          source_name,
        "url":             source_url,
        "extracted_at":    datetime.now().isoformat(),
        "record_count":    int(record_count),
        "output_path":     output_path,
        "transformations": transformations or []
    }
    with open(LINEAGE_FILE, "a") as f:
        f.write(json.dumps(entry) + "\n")
    print(f"[LINEAGE] {source_name} -> {record_count} records -> {output_path}")

print("Lineage logger ready")

Lineage logger ready


---
## Section 1: Web Scraping — Company ESG Claims

We scrape the public sustainability pages of 10 FTSE 100 companies to extract 
self-reported ESG claims. This is our primary dataset — the "claims" side of the 
greenwashing analysis.

**What we extract:**
- Net-zero target year (e.g. "net zero by 2050")
- Emissions reduction % claimed (e.g. "reduce emissions by 50%")
- Third-party certifications (SBTi, CDP, TCFD, ISO14001, RE100)
- Raw text stored in MongoDB for potential NLP analysis

**Why scraping is necessary:** This data does not exist in any structured API.
It only lives as free text on company websites.

**Known limitation:** Several companies (GSK, AstraZeneca, M&S) use JavaScript 
rendering which prevents static HTML scraping. This is documented as a real-world 
data engineering challenge — it would require Selenium/Playwright to resolve, which 
is outside scope for this pipeline.

**Ethics:** 2-second delay between requests, standard browser User-Agent, 
no login walls circumvented, robots.txt respected.

In [ ]:
import io
import pyarrow as pa
import pyarrow.parquet as pq

COMPANIES = [
    {
        
        "company_id": "BP", "company_name": "BP", "sector": "Energy",
        "urls": [
            "https://www.bp.com/en/global/corporate/sustainability.html",
            "https://www.bp.com/en/global/corporate/sustainability/getting-to-net-zero.html",
            "https://en.wikipedia.org/wiki/BP",
            "https://sciencebasedtargets.org/companies-taking-action"
        ]
    },
    {
        
        "company_id": "BARC", "company_name": "Barclays", "sector": "Finance",
        "urls": [
            "https://home.barclays/sustainability/",
            "https://home.barclays/sustainability/esg-resource-hub/",
            "https://home.barclays/investor-relations/reports-and-events/annual-reports/",
            "https://en.wikipedia.org/wiki/Barclays",
            "https://home.barclays/news/press-releases/",
            "https://home.barclays/insights/sustainability-insights/",
            "https://web.archive.org/web/20231015/https://home.barclays/sustainability/net-zero/",
            "https://web.archive.org/web/20231015/https://home.barclays/sustainability/our-strategy/",
            "https://sciencebasedtargets.org/companies-taking-action"
        ]
    },
    {
        
        "company_id": "LLOY", "company_name": "Lloyds", "sector": "Finance",
        "urls": [
            "https://www.lloydsbankinggroup.com/who-we-are/responsible-business.html",
            "https://www.lloydsbankinggroup.com/investors/annual-report.html",
            "https://en.wikipedia.org/wiki/Lloyds_Banking_Group",
            "https://sciencebasedtargets.org/companies-taking-action"
        ]
    },
    {
        
        "company_id": "RIO", "company_name": "Rio Tinto", "sector": "Mining",
        "urls": [
            "https://www.riotinto.com/sustainability",
            "https://www.riotinto.com/sustainability/climate-change",
            "https://en.wikipedia.org/wiki/Rio_Tinto_(corporation)",
            "https://www.riotinto.com/sustainability/climate-change/paris-agreement"
        ]
    },
    {
        
        "company_id": "GSK", "company_name": "GSK", "sector": "Healthcare",
        "urls": [
            "https://www.gsk.com/en-gb/responsibility/environment/",
            "https://www.gsk.com/en-gb/responsibility/environment/net-zero/",
            "https://www.gsk.com/en-gb/responsibility/our-esg-reporting/",
            "https://www.gsk.com/en-gb/responsibility/environment/environmental-data/",
            "https://web.archive.org/web/20231015/https://www.gsk.com/en-gb/responsibility/environment/",
            "https://sciencebasedtargets.org/companies-taking-action"
        ]
    },
    {
        
        "company_id": "AZN", "company_name": "AstraZeneca", "sector": "Healthcare",
        "urls": [
            "https://en.wikipedia.org/wiki/AstraZeneca",
            "https://web.archive.org/web/20231015/https://www.astrazeneca.com/sustainability.html",
            "https://web.archive.org/web/20231015/https://www.astrazeneca.com/sustainability/ambition-zero-carbon.html",
            "https://web.archive.org/web/20231015/https://www.astrazeneca.com/sustainability/esg-reporting.html",
            "https://sciencebasedtargets.org/companies-taking-action"
        ]
    },
    {
        
        "company_id": "MKS", "company_name": "Marks & Spencer", "sector": "Retail",
        "urls": [
            "https://en.wikipedia.org/wiki/Marks_%26_Spencer",
            "https://web.archive.org/web/20231015/https://corporate.marksandspencer.com/sustainability/planet",
            "https://web.archive.org/web/20231015/https://corporate.marksandspencer.com/sustainability/planet/net-zero",
            "https://web.archive.org/web/20231015/https://corporate.marksandspencer.com/sustainability/planet/energy-and-carbon",
            "https://corporate.marksandspencer.com/investors/results-reports-and-presentations",
            "https://sciencebasedtargets.org/companies-taking-action"
        ]
    },
    {
        
        "company_id": "VOD", "company_name": "Vodafone", "sector": "Telecom",
        "urls": [
            "https://en.wikipedia.org/wiki/Vodafone",
            "https://www.vodafone.com/sustainability",
            "https://web.archive.org/web/20231015/https://www.vodafone.com/sustainability/reporting-and-policies",
            "https://sciencebasedtargets.org/companies-taking-action"
        ]
    },
    {
        
        "company_id": "NATG", "company_name": "National Grid", "sector": "Utilities",
        "urls": [
            "https://www.nationalgrid.com/responsibility",
            "https://web.archive.org/web/20231015/https://www.nationalgrid.com/responsibility/environment",
            "https://sciencebasedtargets.org/companies-taking-action"
        ]
    },
    {
        
        "company_id": "SBRY", "company_name": "Sainsburys", "sector": "Retail",
        "urls": [
            "https://www.about.sainsburys.co.uk/sustainability",
            "https://www.about.sainsburys.co.uk/sustainability/plan-for-better/net-zero",
            "https://www.about.sainsburys.co.uk/sustainability/plan-for-better/our-planet",
            "https://en.wikipedia.org/wiki/Sainsbury%27s",
            "https://www.about.sainsburys.co.uk/news",
            "https://www.about.sainsburys.co.uk/sustainability/plan-for-better/net-zero/our-carbon-footprint",
            "https://sciencebasedtargets.org/companies-taking-action"
        ]
    }
]

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

YEAR_PATTERNS = [
    r"net.zero\D{0,30}(20\d{2})",         r"(20\d{2})\D{0,30}net.zero",
    r"carbon.neutral\D{0,30}(20\d{2})",   r"(20\d{2})\D{0,30}carbon.neutral",
    r"zero.carbon\D{0,20}(20\d{2})",      r"net.zero.by.(20\d{2})",
    r"climate.neutral\D{0,30}(20\d{2})",  r"(20\d{2})\D{0,30}climate.neutral",
    r"achieve.net.zero\D{0,20}(20\d{2})", r"zero.emissions\D{0,20}(20\d{2})"
]

PCT_PATTERNS = [
    r"reduc\w+\D{0,15}(\d{1,3})\s*%",   r"(\d{1,3})\s*%\D{0,15}reduc",
    r"cut\D{0,15}(\d{1,3})\s*%",         r"(\d{1,3})\s*%\D{0,20}emission",
    r"emissions\D{0,20}(\d{1,3})\s*%",   r"decarboni\w+\D{0,15}(\d{1,3})\s*%",
    r"(\d{1,3})\s*%\D{0,20}carbon",      r"lower\D{0,15}(\d{1,3})\s*%",
    r"(\d{1,3})\s*% absolute reduction",  r"scope.1.and.2\D{0,20}(\d{1,3})\s*%"
]

CERT_KEYWORDS = {
    "SBTi":     ["science based targets", "sbti", "science-based targets"],
    "CDP":      ["cdp", "carbon disclosure project"],
    "TCFD":     ["tcfd", "task force on climate"],
    "ISO14001": ["iso 14001", "iso14001"],
    "RE100":    ["re100", "100% renewable energy"],
    "PAS2060":  ["pas2060", "pas 2060"]
}

def extract_from_text(text):
    net_zero_year, reduction_pct, certs = None, None, []

    for pat in YEAR_PATTERNS:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            yr = int(m.group(1))
            
            if 2025 <= yr <= 2060:
                net_zero_year = yr
                break

    for pat in PCT_PATTERNS:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            val = float(m.group(1))
            # Normalise to 0-100 scale:
            # Only scale true fractions (e.g. 0.5 means 50%). Values >= 1 are
            # already literal percentage points — do NOT multiply (avoids 5% -> 50%)
            if 0 < val < 1:
                val = round(val * 100, 1)   # 0.5 -> 50.0
            # Accept only meaningful headline claims (10-99%)
            if 10 <= val <= 99:
                reduction_pct = val; break

    for cert, kws in CERT_KEYWORDS.items():
        if any(kw in text.lower() for kw in kws):
            certs.append(cert)

    return net_zero_year, reduction_pct, certs

def scrape_single_url(url):
    try:
        time.sleep(2)
        resp = requests.get(url, headers=HEADERS, timeout=20)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        tags = soup.find_all(["p","h1","h2","h3","li","span","div"])
        paras = [t.get_text(strip=True) for t in tags if len(t.get_text(strip=True)) > 30]
        return " ".join(paras[:300])
    except Exception as e:
        return None  # JS-rendered or blocked page

def scrape_company_multi_url(company: dict) -> dict:
    result = {
        "company_id":            company["company_id"],
        "company_name":          company["company_name"],
        "sector":                company["sector"],
        "net_zero_target_year":  None,
        "reduction_pct_claimed": None,
        "certifications":        set(),
        "raw_text":              "",
        "urls_scraped":          [],
        "urls_successful":       0,
        "source_url":            company["urls"][0],
        "scraped_at":            datetime.now().isoformat()
    }

    for url in company["urls"]:
        if (result["net_zero_target_year"] is not None and
            result["reduction_pct_claimed"] is not None and
            len(result["certifications"]) >= 2):
            print(f"      - All fields found — stopping early ")
            break

        label = ("wayback→" + url.split("https://")[-1].split("/")[0]
                 if "web.archive.org" in url
                 else "/".join(url.split("/")[2:4]))
        print(f"      - {label}...", end=" ")
        text = scrape_single_url(url)

        if text:
            year, pct, certs = extract_from_text(text)
            result["urls_successful"] += 1
            result["raw_text"]        += " " + text[:800]
            result["urls_scraped"].append(url)

            if result["net_zero_target_year"] is None and year:
                result["net_zero_target_year"] = year
                print(f"year={year} ", end=" ")
            if result["reduction_pct_claimed"] is None and pct:
                result["reduction_pct_claimed"] = pct
                print(f"pct={pct}% ", end=" ")
            if certs:
                new = set(certs) - result["certifications"]
                if new:
                    result["certifications"].update(new)
                    print(f"certs={list(new)} ", end=" ")
            print()
        else:
            print("WARNING JS/blocked")

    result["certifications"] = ",".join(sorted(result["certifications"])) \
                                if result["certifications"] else None
    result["raw_text"]       = result["raw_text"].strip()[:5000]
    return result

print(f" Total URLs: {sum(len(c['urls']) for c in COMPANIES)}")

In [ ]:
print("Starting multi-URL web scraping...\n")
print("=" * 60)

scraped_results = []
for company in COMPANIES:
    print(f"\n   {company['company_name']} ({len(company['urls'])} URLs):")
    result = scrape_company_multi_url(company)
    scraped_results.append(result)
    print(f"   RESULT -> year={result['net_zero_target_year']} | "
          f"pct={result['reduction_pct_claimed']} | "
          f"certs={result['certifications']} | "
          f"URLs ok: {result['urls_successful']}/{len(company['urls'])}")

claims_df = pd.DataFrame(scraped_results)

print(f"\n{'=' * 60}")
print(f"SCRAPING COMPLETE")
print(f"  Net zero year : {claims_df['net_zero_target_year'].notna().sum()}/10")
print(f"  Reduction %   : {claims_df['reduction_pct_claimed'].notna().sum()}/10")
print(f"  Certifications: {claims_df['certifications'].notna().sum()}/10")
print(f"  Avg URLs ok   : {claims_df['urls_successful'].mean():.1f}/4")
print(f"{'=' * 60}")


claims_df[["company_name","net_zero_target_year","reduction_pct_claimed",
           "certifications","urls_successful"]]

### Store Scraped Data
Raw text goes to MongoDB (unstructured, variable length — ideal for NoSQL document store).
Structured fields go to Parquet (columnar format, optimised for Spark and DuckDB queries).
This dual-storage pattern is intentional — different storage engines serve different query patterns.

In [ ]:

# ── MongoDB: store raw text (NoSQL — ideal for unstructured variable-length content)
db = None  # will be set if MongoDB is available
try:
    mongo_client = pymongo.MongoClient(MONGO_URI, serverSelectionTimeoutMS=3000)
    mongo_client.server_info()
    db = mongo_client["greenwashing"]
    db["company_claims_raw"].drop()
    db["company_claims_raw"].insert_many(scraped_results)
    print(f" MongoDB: {len(scraped_results)} records -> greenwashing.company_claims_raw")
except Exception as e:
    print(f"WARNING MongoDB unavailable (continuing without it): {e}")

# ── Parquet: structured fields only, saved via BytesIO (Windows permission safe)
os.makedirs("data/raw", exist_ok=True)
claims_structured = claims_df.drop(columns=["raw_text","urls_scraped"], errors="ignore")
buffer = io.BytesIO()
pq.write_table(pa.Table.from_pandas(claims_structured, preserve_index=False), buffer)
with open("data/raw/company_claims.parquet", "wb") as f:
    f.write(buffer.getvalue())
print(f" Parquet: saved -> data/raw/company_claims.parquet")
print(f"   Columns: {list(claims_structured.columns)}")

log_lineage(
    source_name  = "FTSE100 Multi-URL Web Scraping (4 URLs per company)",
    source_url   = "Sustainability pages + Climate pages + Wikipedia + Press releases",
    record_count = len(claims_df),
    output_path  = "data/raw/company_claims.parquet",
    transformations = [
        "4 URLs per company: sustainability, climate subpage, Wikipedia, press releases",
        "8 regex patterns for net-zero year extraction",
        "8 regex patterns for reduction % extraction",
        "6 certification types matched (SBTi, CDP, TCFD, ISO14001, RE100, PAS2060)",
        "Findings merged per company — all URLs feed one record",
        "Raw text stored in MongoDB, structured fields in Parquet"
    ]
)
print(f" Saved -> data/raw/company_claims.parquet ({os.path.getsize('data/raw/company_claims.parquet'):,} bytes)")
claims_df[["company_name","net_zero_target_year","reduction_pct_claimed","certifications","urls_successful"]]


## Section 1b: SQLite — Relational Database Storage

To satisfy the "SQL/NoSQL/Graph" storage requirement, we write the structured
company claims into a **SQLite relational database**. SQLite is a serverless
SQL engine — zero configuration, built into Python, and appropriate here
because our structured claims data has a fixed schema ideal for relational storage.

**Why SQLite here vs MongoDB?**
MongoDB stores the variable-length raw text (unstructured). SQLite stores the
structured, typed fields (net-zero year, reduction %, certifications) — a clean
separation of concerns between NoSQL and SQL storage layers.

**Tables created:**
- `company_claims` — one row per company, structured ESG claims
- `sector_summary` — aggregated sector-level view (demonstrates SQL DDL + aggregation)

In [ ]:
import sqlite3

# ── SQLite: Relational database for structured claims ──
sqlite_path = "data/greenwashing.db"
os.makedirs("data", exist_ok=True)

conn = sqlite3.connect(sqlite_path)
cursor = conn.cursor()

# Explicit DDL schema definition — demonstrates relational design
cursor.executescript("""
   DROP TABLE IF EXISTS company_claims;
   DROP TABLE IF EXISTS sector_summary;

   CREATE TABLE company_claims (
       company_id              TEXT PRIMARY KEY,
       company_name            TEXT NOT NULL,
       sector                  TEXT NOT NULL,
       net_zero_target_year    INTEGER,
       reduction_pct_claimed   REAL,
       certifications          TEXT,
       urls_successful         INTEGER,
       scraped_at              TEXT
   );

   CREATE TABLE sector_summary (
       sector                  TEXT PRIMARY KEY,
       company_count           INTEGER,
       avg_reduction_claimed   REAL,
       certified_count         INTEGER
   );
""")

# Insert company_claims rows
for _, row in claims_structured.iterrows():
   cursor.execute("""
       INSERT OR REPLACE INTO company_claims
       VALUES (?, ?, ?, ?, ?, ?, ?, ?)
   """, (
       row.get("company_id"),
       row.get("company_name"),
       row.get("sector"),
       row.get("net_zero_target_year") if pd.notna(row.get("net_zero_target_year")) else None,
       row.get("reduction_pct_claimed") if pd.notna(row.get("reduction_pct_claimed")) else None,
       row.get("certifications"),
       int(row.get("urls_successful", 0)),
       row.get("scraped_at")
   ))

# Insert sector_summary using SQL aggregation
cursor.execute("""
   INSERT INTO sector_summary
   SELECT
       sector,
       COUNT(*) AS company_count,
       ROUND(AVG(reduction_pct_claimed), 1) AS avg_reduction_claimed,
       COUNT(CASE WHEN certifications IS NOT NULL THEN 1 END) AS certified_count
   FROM company_claims
   GROUP BY sector
""")

conn.commit()

# Verify — print both tables
print("=== SQLite: company_claims TABLE ===")
result = pd.read_sql("SELECT * FROM company_claims", conn)
print(result[["company_name","sector","net_zero_target_year","reduction_pct_claimed","certifications"]].to_string(index=False))

print("\n=== SQLite: sector_summary TABLE ===")
result2 = pd.read_sql("SELECT * FROM sector_summary", conn)
print(result2.to_string(index=False))

# Show schema via PRAGMA — explicit schema evidence for marking
print("\n=== SQLite SCHEMA: company_claims ===")
schema = pd.read_sql("PRAGMA table_info(company_claims)", conn)
print(schema[["name","type","notnull","pk"]].to_string(index=False))

conn.close()

log_lineage(
   source_name     = "SQLite Relational Database",
   source_url      = sqlite_path,
   record_count    = len(claims_structured),
   output_path     = sqlite_path,
   transformations = [
       "DDL: CREATE TABLE company_claims with typed schema",
       "DDL: CREATE TABLE sector_summary",
       "INSERT from scraped claims DataFrame",
       "SQL aggregation: sector_summary populated via GROUP BY"
   ]
)
print(f"\n SQLite saved -> {sqlite_path}")

## Section 2: The Guardian API — News Articles

The Guardian provides a free open API (api-key=test) requiring no registration.
We use it to collect journalism covering each company's ESG and climate record.

**Why The Guardian specifically?**
The Guardian is the UK's leading investigative journalism outlet for climate and 
corporate accountability reporting. For UK-listed FTSE 100 companies, it provides 
higher quality and more relevant coverage than general news aggregators.

**Search strategy:** 3 queries per company targeting different angles:
1. `{company} greenwashing` — direct greenwashing allegations
2. `{company} emissions climate` — climate performance reporting  
3. `{company} ESG sustainability` — ESG coverage

Articles are deduplicated by URL to avoid counting the same article twice.

In [ ]:
def fetch_guardian_articles(company_name: str) -> list:
    url = "https://content.guardianapis.com/search"
    records = []
    queries = [f"{company_name} greenwashing",
               f"{company_name} emissions climate",
               f"{company_name} ESG sustainability"]

    for query in queries:
        try:
            time.sleep(1)
            resp = requests.get(url, params={
                "q": query, "page-size": 10,
                "show-fields": "headline,trailText,byline,wordcount",
                "api-key": GUARDIAN_API_KEY or "test"
            }, timeout=15)
            resp.raise_for_status()
            for art in resp.json().get("response", {}).get("results", []):
                records.append({
                    "company_name": company_name,
                    "headline":     art.get("fields", {}).get("headline", art.get("webTitle")),
                    "snippet":      art.get("fields", {}).get("trailText"),
                    "byline":       art.get("fields", {}).get("byline"),
                    "published_at": art.get("webPublicationDate"),
                    "section":      art.get("sectionName"),
                    "url":          art.get("webUrl"),
                    "api_source":   "Guardian Open API",
                    "extracted_at": datetime.now().isoformat()
                })
        except Exception as e:
            print(f"  Guardian ({query[:25]}): {e}")

    seen, unique = set(), []
    for r in records:
        if r["url"] not in seen:
            seen.add(r["url"]); unique.append(r)
    return unique

print("Fetching Guardian articles...")
all_articles = []
for company in COMPANIES:
    print(f"  {company['company_name']}...", end=" ")
    arts = fetch_guardian_articles(company["company_name"])
    all_articles.extend(arts)
    print(f"{len(arts)} articles")
    time.sleep(1)

news_df = pd.DataFrame(all_articles)
news_df["published_at"] = pd.to_datetime(news_df["published_at"], errors="coerce")
news_df["company_name"] = news_df["company_name"].str.strip()

# MongoDB
if db is not None:
    try:
        db["news_articles_raw"].drop()
        db["news_articles_raw"].insert_many(all_articles)
        print(f"MongoDB: {len(all_articles)} articles saved")
    except Exception as e:
        print(f"MongoDB news insert failed: {e}")

news_df.to_parquet("data/raw/news_articles.parquet", index=False)
log_lineage("The Guardian Open API", "content.guardianapis.com/search",
            len(news_df), "data/raw/news_articles.parquet",
            transformations=["3 queries per company", "Deduplicated by URL",
                             "Stored in MongoDB and Parquet"])

print(f"Total articles: {len(news_df)}")
news_df[["company_name","headline","section","published_at"]].head(10)

## Section 3: Reddit Public JSON API — Community Sentiment

Reddit exposes a public `.json` endpoint on every search page, requiring no 
authentication. We use this to capture community-level sentiment — real public 
opinion about these companies' environmental records.

**Why Reddit?**
Reddit discussions represent organic public sentiment from informed communities.
Subreddits like r/investing and r/environment contain detailed analysis from users 
who follow these companies closely. High upvote scores and comment counts indicate 
posts the community found significant.

**Search strategy:**
- Global Reddit search (all subreddits) — 25 posts per company
- 8 targeted subreddits confirmed to be publicly accessible
- 4-second delay between requests to avoid rate limiting (HTTP 429)
- 5-second delay between companies to reset rate limit window
- Deduplicated by post_id to avoid counting reposts

In [ ]:
def fetch_reddit_posts(company_name: str) -> list:
    records = []
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

    def parse_posts(posts, source):
        results = []
        for post in posts:
            d = post.get("data", {})
            results.append({
                "company_name": company_name,
                "post_id":      d.get("id"),
                "title":        d.get("title"),
                "text":         d.get("selftext", "")[:500],
                "score":        d.get("score"),
                "upvote_ratio": d.get("upvote_ratio"),
                "num_comments": d.get("num_comments"),
                "subreddit":    d.get("subreddit"),
                "url":          f"https://reddit.com{d.get('permalink','')}",
                "created_utc":  datetime.utcfromtimestamp(d.get("created_utc",0)).isoformat(),
                "api_source":   "Reddit Public JSON",
                "extracted_at": datetime.now().isoformat()
            })
        return results

    try:
        time.sleep(3)
        resp = requests.get("https://www.reddit.com/search.json", headers=headers,
            params={"q": f"{company_name} greenwashing OR emissions OR ESG OR sustainability OR climate",
                    "sort": "relevance", "limit": 25, "t": "all"}, timeout=15)
        resp.raise_for_status()
        posts = resp.json().get("data", {}).get("children", [])
        records.extend(parse_posts(posts, "global"))
        print(f"global:{len(posts)}", end=" ")
    except Exception as e:
        print(f"global:skip({type(e).__name__})", end=" ")

    for sub in ["investing", "environment", "sustainability", "CorporateMalfeasance",
                "anticonsumption", "collapse", "worldnews", "Economics"]:
        try:
            time.sleep(4)
            resp = requests.get(f"https://www.reddit.com/r/{sub}/search.json",
                headers=headers,
                params={"q": f"{company_name} greenwashing OR emissions OR ESG OR climate",
                        "restrict_sr": "true", "sort": "relevance", "limit": 10, "t": "all"},
                timeout=15)
            resp.raise_for_status()
            posts = resp.json().get("data", {}).get("children", [])
            records.extend(parse_posts(posts, sub))
            print(f"r/{sub}:{len(posts)}", end=" ")
        except Exception as e:
            print(f"r/{sub}:skip({type(e).__name__})", end=" ")

    seen, unique = set(), []
    for r in records:
        if r["post_id"] not in seen:
            seen.add(r["post_id"]); unique.append(r)
    return unique

print("Fetching Reddit posts...")
all_reddit = []
for company in COMPANIES:
    print(f"  {company['company_name']}: ", end="")
    posts = fetch_reddit_posts(company["company_name"])
    all_reddit.extend(posts)
    print(f"-> {len(posts)} unique posts")
    time.sleep(5)

reddit_df = pd.DataFrame(all_reddit)
reddit_df["score"]        = pd.to_numeric(reddit_df["score"],        errors="coerce").fillna(0).astype(int)
reddit_df["num_comments"] = pd.to_numeric(reddit_df["num_comments"], errors="coerce").fillna(0).astype(int)
reddit_df["upvote_ratio"] = pd.to_numeric(reddit_df["upvote_ratio"], errors="coerce").fillna(0.0)
reddit_df["company_name"] = reddit_df["company_name"].str.strip()
reddit_df = reddit_df[reddit_df["title"].notna()]

# MongoDB
if db is not None:
    try:
        db["reddit_posts_raw"].drop()
        db["reddit_posts_raw"].insert_many(reddit_df.to_dict("records"))
        print(f"MongoDB: {len(reddit_df)} posts saved")
    except Exception as e:
        print(f"MongoDB reddit insert failed: {e}")

reddit_df.to_parquet("data/raw/reddit_posts.parquet", index=False)
log_lineage("Reddit Public JSON API",
            "reddit.com (global + 8 subreddits)",
            len(reddit_df), "data/raw/reddit_posts.parquet",
            transformations=["Global search + 8 subreddits", "Deduplicated by post_id",
                             "Stored in MongoDB and Parquet"])

print(f"Total posts: {len(reddit_df)}")
print(reddit_df["company_name"].value_counts().to_string())

## Section 4: Our World in Data — Verified UK CO2 Emissions

Our World in Data (OWID) publishes a comprehensive, peer-reviewed CO2 dataset 
maintained by climate researchers. It is freely available as a CSV on GitHub.

**Why this source?**
This provides the "ground truth" emissions data to compare against company claims.
If a company claims to have reduced emissions by 50% but national data shows the 
sector's emissions are flat or rising, that is a potential greenwashing signal.

**What we extract:** UK-only records from 2000 onwards including total CO2, 
CO2 per capita, total GHG, and fuel-type breakdowns (coal, oil, gas).

In [ ]:
def fetch_owid_co2():
    url = "https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv"
    try:
        df = pd.read_csv(url)
        uk = df[(df["country"] == "United Kingdom") & (df["year"] >= 2000)].copy()
        uk = uk[["country","year","co2","co2_per_capita","ghg_per_capita",
                 "energy_per_gdp","total_ghg","coal_co2","oil_co2","gas_co2"]].reset_index(drop=True)
        uk["source"]       = "Our World in Data CO2 Dataset"
        uk["extracted_at"] = datetime.now().isoformat()
        return uk
    except Exception as e:
        print(f"  ERROR OWID error: {e}")
        return pd.DataFrame()

print("Fetching OWID CO2 data...")
owid_df = fetch_owid_co2()
owid_df.to_parquet("data/raw/owid_co2.parquet", index=False)

log_lineage("Our World in Data — UK CO2 Dataset",
            "github.com/owid/co2-data",
            len(owid_df), "data/raw/owid_co2.parquet",
            transformations=["Filtered: United Kingdom only",
                             "Filtered: year >= 2000",
                             "Selected 10 emissions columns"])

print(f" {len(owid_df)} records | {owid_df['year'].min()}–{owid_df['year'].max()}")
owid_df.tail(8)

## Section 5: Graph Database — Company Relationship Network

We model the relationships between companies, sectors, and certifications 
as a directed graph using NetworkX.

**Why a graph database?**
Graph databases excel at relationship queries that are inefficient or impossible 
in flat tables. For our pipeline, the graph answers questions like:
- Which certifications are shared across multiple sectors?
- Which companies are isolated (no certifications, no sector peers)?
- How connected is the ESG certification network?

**Graph structure:**
- **Nodes:** Companies (10), Sectors (5), Certifications (variable)
- **Edges:** company -> sector (operates_in), company -> certification (certified_by)
- **Node attributes:** company_id, net_zero_year, reduction_pct stored on each node

The graph is exported as GEXF format — the standard for graph databases, 
openable in Gephi for visual network analysis.

In [ ]:
G = nx.DiGraph()

for _, row in claims_df.iterrows():
    company = row["company_name"]
    sector  = row["sector"]

    G.add_node(company, type="company",
               company_id    = row["company_id"],
               net_zero_year = str(row["net_zero_target_year"]),
               reduction_pct = str(row["reduction_pct_claimed"]))

    G.add_node(sector, type="sector")
    G.add_edge(company, sector, relationship="operates_in")

    if row["certifications"] and str(row["certifications"]) != "None":
        for cert in str(row["certifications"]).split(","):
            cert = cert.strip()
            if cert:
                G.add_node(cert, type="certification")
                G.add_edge(company, cert, relationship="certified_by")

nx.write_gexf(G, "data/processed/company_graph.gexf")

# Print summary
node_types = {}
for _, attrs in G.nodes(data=True):
    t = attrs.get("type","unknown")
    node_types[t] = node_types.get(t,0) + 1

print(f" Graph database saved -> data/processed/company_graph.gexf")
print(f"\n  Nodes: {G.number_of_nodes()}")
for t,c in node_types.items():
    print(f"    {t}: {c}")
print(f"\n  Edges: {G.number_of_edges()}")
for src, tgt, attrs in G.edges(data=True):
    print(f"    {src} -> {tgt} [{attrs.get('relationship')}]")

log_lineage("NetworkX Graph — Company ESG Relationships",
            "data/raw/company_claims.parquet",
            G.number_of_nodes(), "data/processed/company_graph.gexf",
            transformations=["company→sector edges", "company→certification edges",
                             "Node attributes: net_zero_year, reduction_pct",
                             "Exported as GEXF"])